### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [89]:
# Test 1. train on v2 improves base accuracy why? We want to break down the v2 trainer 
#         expose its granular implementation and validate we are actually just training for p(s | a)
# Test 2. for v3, the corrupted abstraction might not be different from clean abstration due to stuttering, 
#         we want to expose the training codes for v3, and debug it in the notebook, too

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper

from sorl.trainer_ablate import SoRLTrainerv2, SoRLTrainerv3
from sorl.trainer_ablate import SoRLConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load a trained ckpt
from train_ablate_sanity import load_checkpoint

teacher, tokenizer = load_checkpoint("Qwen/Qwen3-1.7B", 1000, "checkpoints/sorlv3_gsm8k_20251015_163120", "cpu")

In [3]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
# ============================================================
# SoRL v3 Training Kernel — all logic inline, easy to hack
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, os, json
from sorl.sorl_trainer import sorl_search, corrupt_abstract_tokens, ortho_loss
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# ---- Config (edit freely) ----
cfg = dict(
    # Model
    model_name   = "Qwen/Qwen3-0.6B",
    abs_vocab    = 128,

    # Data
    dataset      = "gsm8k",
    max_length   = 256,
    batch_size   = 2,

    # SoRL search
    K            = 4,
    num_rollouts = 4,
    max_iters    = 2,
    temperature  = 1.0,
    mem_span_abs = 1792,
    mem_span_traj= 1792,

    # Corruption
    corrupt_method = "shuffle",   # "shuffle" or "noise"
    corrupt_ratio  = 0.3,
    n_corrupt      = 1,         # number of fresh corruptions per batch (average their loss)
    

    # Loss weights
    alpha_traj       = 1.0,     # p(s|a)
    alpha_contrastive= 100.0,     # hinge loss
    alpha_abs        = 0.5,     # p(a|s)
    alpha_ortho      = 0.0,
    gamma            = 0.5,     # hinge margin

    # Optimizer
    lr           = 1e-5,
    emb_lr_mult  = 1.0,
    weight_decay = 0.01,
    max_grad_norm= 1.0,
    warmup_steps = 50,
    cooldown_frac= 0.4,

    # Training
    num_epochs   = 3,
    grad_accum   = 4,
    log_every    = 10,
    eval_every   = 99999,
    eval_samples = 100,
    save_every   = 99999,
    output_dir   = "./ckpt/v3_kernel",
)

print("Config:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

Config:
  model_name: Qwen/Qwen3-0.6B
  abs_vocab: 128
  dataset: gsm8k
  max_length: 256
  batch_size: 2
  K: 4
  num_rollouts: 4
  max_iters: 2
  temperature: 1.0
  mem_span_abs: 1792
  mem_span_traj: 1792
  corrupt_method: shuffle
  corrupt_ratio: 0.3
  n_corrupt: 1
  alpha_traj: 1.0
  alpha_contrastive: 100.0
  alpha_abs: 0.5
  alpha_ortho: 0.0
  gamma: 0.5
  lr: 1e-05
  emb_lr_mult: 1.0
  weight_decay: 0.01
  max_grad_norm: 1.0
  warmup_steps: 50
  cooldown_frac: 0.4
  num_epochs: 3
  grad_accum: 4
  log_every: 10
  eval_every: 99999
  eval_samples: 100
  save_every: 99999
  output_dir: ./ckpt/v3_kernel


In [5]:
# ============================================================
# Model + Data + Optimizer setup
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SorlModelWrapper.from_pretrained(cfg["model_name"], abstract_vocab_size_list=[cfg["abs_vocab"]])
model = model.to(device).train()
tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
pad_token_id = tokenizer.pad_token_id
base_vocab = int(model.vocab_sizes[0].item())
total_vocab = int(model.vocab_sizes.sum().item())

train_ds = get_dataset(cfg["dataset"], split="train", tokenizer=tokenizer, max_length=cfg["max_length"])
val_ds   = get_dataset(cfg["dataset"], split="test",  tokenizer=tokenizer, max_length=cfg["max_length"])
dl = torch.utils.data.DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate_fn, num_workers=0)

# Separate param groups for embedding LR multiplier
emb_params, other_params = [], []
for name, p in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        emb_params.append(p)
    else:
        other_params.append(p)
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": cfg["lr"]},
    {"params": emb_params,   "lr": cfg["lr"] * cfg["emb_lr_mult"]},
], weight_decay=cfg["weight_decay"])

total_steps = len(dl) * cfg["num_epochs"] // cfg["grad_accum"]
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Steps/epoch: {len(dl)} | Total steps: {total_steps}")
print(f"Effective batch: {cfg['batch_size'] * cfg['grad_accum']}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Train: 7473 | Val: 1319 | Steps/epoch: 3737 | Total steps: 2802
Effective batch: 8


In [ ]:
# ============================================================
# Helper functions
# ============================================================

def get_lr(step):
    """Warmup + linear cooldown."""
    ws = cfg["warmup_steps"]
    if step < ws:
        return cfg["lr"] * step / max(ws, 1)
    progress = (step - ws) / max(total_steps - ws, 1)
    if progress < 1 - cfg["cooldown_frac"]:
        return cfg["lr"]
    w = (1 - progress) / cfg["cooldown_frac"]
    return cfg["lr"] * (w * 1.0 + (1 - w) * 0.1)


def build_masks(data, attn_mask, prompt_len, base_vocab):
    """Build traj_mask (NL positions) and abs_mask (abstract positions) for expanded sequence."""
    shift_attn = attn_mask[..., 1:].contiguous().clone()
    seq_idx = torch.arange(shift_attn.size(1), device=data.device).unsqueeze(0)
    shift_attn[seq_idx < (prompt_len.unsqueeze(1) - 1)] = 0
    levels = (data >= base_vocab).long()[:, 1:]
    traj_mask = (levels == 0).float() * shift_attn.float()
    abs_mask  = (1 - traj_mask) * shift_attn.float()
    return traj_mask, abs_mask


def compute_traj_loss_from_logits(shift_logits, data, traj_mask, base_vocab):
    """CE on NL positions with abstract logits masked to -inf."""
    tl = shift_logits.clone()
    tl[..., base_vocab:] = -float("inf")
    safe = data[..., 1:].clone()
    safe[~traj_mask.bool()] = 0
    losses = nn.CrossEntropyLoss(reduction='none')(
        tl.view(-1, tl.size(-1)), safe.view(-1)
    ).view(data.shape[0], -1) * traj_mask
    return losses.sum() / traj_mask.sum().clamp(min=1)


def compute_abs_loss_from_logits(shift_logits, data, abs_mask, base_vocab):
    """CE on abstract positions with base vocab masked to -inf."""
    al = shift_logits.clone()
    al[..., :(base_vocab + 1)] = -float("inf")
    safe = data[..., 1:].clone()
    safe[~abs_mask.bool()] = base_vocab + 1
    losses = nn.CrossEntropyLoss(reduction='none')(
        al.view(-1, al.size(-1)), safe.view(-1)
    ).view(data.shape[0], -1) * abs_mask
    return losses.sum() / abs_mask.sum().clamp(min=1)


def compute_corrupted_traj_loss(model, corrupted_data, attn_mask, traj_mask, base_vocab):
    """Forward corrupted sequence (no grad), return traj_loss on NL positions."""
    with torch.no_grad():
        out = model(input_ids=corrupted_data, attention_mask=attn_mask,
                    memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"])
        s = out.logits[..., :-1, :].contiguous()
        return compute_traj_loss_from_logits(s, corrupted_data, traj_mask, base_vocab)


print("Helpers defined: build_masks, compute_traj_loss_from_logits, compute_abs_loss_from_logits, compute_corrupted_traj_loss")

In [30]:
INNER_LOOP_COUNT = 100

In [ ]:
# ============================================================
# Training loop
# ============================================================
os.makedirs(cfg["output_dir"], exist_ok=True)
history = {"step": [], "loss": [], "base_loss": [], "traj_loss": [],
           "hinge_loss": [], "abs_loss": [], "ortho_loss": [], "lr": []}

model.train()
global_step = 0
t_start = time.time()

for epoch in range(cfg["num_epochs"]):
    for batch_idx, batch in enumerate(dl):
        input_ids     = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        prompt_len    = batch["prompt_len"].to(device)

        # ---- LR schedule ----
        lr = 1e-5
        optimizer.param_groups[0]["lr"] = lr
        optimizer.param_groups[1]["lr"] = lr * cfg["emb_lr_mult"]

        # ---- 1. Base traj loss (logging only) ----
        with torch.no_grad():
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            si = torch.arange(labels.size(1), device=device).unsqueeze(0)
            labels[si < prompt_len.unsqueeze(1)] = -100
            out = model(input_ids=input_ids, attention_mask=attention_mask,
                        memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"])
            lg = out.logits.clone()
            lg[:, :, base_vocab:] = -float("inf")
            base_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                lg[:, :-1].contiguous().view(-1, lg.size(-1)),
                labels[:, 1:].contiguous().view(-1)
            )
            del out, lg

        # ---- 2. SoRL search (no grad) ----
        with torch.no_grad():
            best_data, _, _, exp_attn, exp_pl = sorl_search(
                teacher, input_ids, attention_mask, prompt_len, pad_token_id,
                n=cfg["num_rollouts"], K=cfg["K"], max_iterations=cfg["max_iters"],
                memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"],
                temperature=cfg["temperature"],
            )

        traj_mask, abs_mask = build_masks(best_data, exp_attn, exp_pl, base_vocab)
    
        # Inner Loop starts here
        for ilc in range(INNER_LOOP_COUNT): 
            
            corrupted = corrupt_abstract_tokens(
                best_data, base_vocab, total_vocab,
                method=cfg["corrupt_method"], corrupt_ratio=cfg["corrupt_ratio"],
            ) 
            n_diff = (corrupted != best_data).sum().item()

            corrupt_traj = compute_corrupted_traj_loss(model, corrupted, exp_attn, traj_mask, base_vocab)
        
            outputs = model(input_ids=best_data, attention_mask=exp_attn,
                            memory_span_abs=cfg["mem_span_abs"], memory_span_traj=cfg["mem_span_traj"])
            shift_logits_clean = outputs.logits[..., :-1, :].contiguous()
    
            traj_loss = compute_traj_loss_from_logits(shift_logits_clean, best_data, traj_mask, base_vocab)            
            abs_loss  = compute_abs_loss_from_logits(shift_logits_clean, best_data, abs_mask, base_vocab)
            hinge_loss = (cfg["gamma"] + traj_loss - corrupt_traj).clamp(min=0)
            
            print(f"- [{ilc+1}/{INNER_LOOP_COUNT} inner loop hinge loss with {n_diff} mutations]: ", hinge_loss.item())
    
            # 2. SCALE THE LOSS FOR ACCUMULATION
            loss = (
                cfg["alpha_traj"]        * traj_loss
              + cfg["alpha_contrastive"] * hinge_loss
              + cfg["alpha_abs"]         * abs_loss
            ) / cfg["grad_accum"]
            
            loss.backward()
            
            del outputs, shift_logits_clean

            # 3. USE A SIMPLE, MONOTONIC COUNTER FOR OPTIMIZATION
            accum_steps += 1
            if accum_steps % cfg["grad_accum"] == 0: 
                print("[Optimizer Step taken]")
                if cfg["max_grad_norm"] > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
         

        # ---- Logging ----
        if (batch_idx + 1) % cfg["log_every"] == 0:
            total_loss = loss.item() * cfg["grad_accum"]
            elapsed = time.time() - t_start
            frac = max(global_step, 1) / max(total_steps, 1)
            eta = elapsed / frac * (1 - frac) if frac > 0 else 0
            eta_m, eta_s = divmod(int(eta), 60)
            eta_h, eta_m = divmod(eta_m, 60)

            print(f"ep {epoch + (batch_idx+1)/len(dl):.3f}/{cfg['num_epochs']} "
                  f"| eta {eta_h}h{eta_m:02d}m "
                  f"| loss={total_loss:.4f} base={base_loss.item():.4f} "
                  f"traj={traj_loss.item():.4f} hinge={hinge_loss.item():.4f} "
                  f"abs={abs_loss.item():.4f} ortho={orth_loss.item():.4f} "
                  f"| lr={lr:.2e}")

            history["step"].append(global_step)
            history["loss"].append(total_loss)
            history["base_loss"].append(base_loss.item())
            history["traj_loss"].append(traj_loss.item())
            history["hinge_loss"].append(hinge_loss.item())
            history["abs_loss"].append(abs_loss.item())
            history["ortho_loss"].append(orth_loss.item())
            history["lr"].append(lr)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ---- Eval ----
        if global_step > 0 and global_step % cfg["eval_every"] == 0:
            model.eval()
            res = evaluate_accuracy(model, tokenizer, val_ds, device, cfg["eval_samples"])
            print(f"  === Eval step {global_step}: {res} ===")
            model.train()

    print(f"=== Epoch {epoch + 1} complete ===")

# Save history
with open(os.path.join(cfg["output_dir"], "history.json"), "w") as f:
    json.dump(history, f)
print(f"Training complete! History saved to {cfg['output_dir']}/history.json")

- [1/100 inner loop hinge loss with 17 mutations]:  0.44238901138305664
[Optimizer Step taken]
- [2/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [3/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [4/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
- [5/100 inner loop hinge loss with 17 mutations]:  0.3889656066894531
[Optimizer Step taken]
- [6/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [7/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [8/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
- [9/100 inner loop hinge loss with 17 mutations]:  0.295021653175354
[Optimizer Step taken]
- [10/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [11/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [12/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
- [13/100 inner loop hinge loss with 17 mutations]:  0.24538421630859375
[Op

KeyboardInterrupt: 

In [ ]:
# Observation #1. Inner loop + dynamic corrupted tokens + hinge loss optimization (allow degrading p(s | a_corrupt)) => zero out hinge loss in 10 steps

# Observation #2. Inner loop + dynamic corrupted tokens + combo loss optimization (allow degrading p(s | a_corrupt)) => zero out hinge loss in 10 steps

# Observation #3. Inner loop + dynamic corrupted tokens + combo loss optimization (disallow degrading p(s | a_corrupt)) => hinge loss decreases, often zero out in 20 steps,
#                 then increase a bit, but overall decrease

# Observation #4. Inner loop + static corrupted tokens + combo loss optimization (disallow degrading p(s | a_corrupt)) => hinge loss decrease, then increase above prev levels

# Observation #5. Inner loop + static corrupted tokens + hinge loss optimization (disallow degrading p(s | a_corrupt)) => hinge loss decrease then increase above prev levels
#                 [Why?? If we are optimizing a single objective with fixed input, how can it decrease then increase? Is this a bug?]

# Observation #6. Inner loop + dynamic corrupted tokens + hinge loss optimization (disallow degrading p(s | a_corrupt)) => hinge loss decrease, but not consistently
#                 [Hingle loss conflics reduces with dynamic corrupted tokens]



